# EdgeForge - Part 2: Hardware-Aware Neural Architecture Search (CIFAR-10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sudoHasaan/edgeforge/blob/main/notebooks/02_hardware_aware_nas.ipynb)

Pipeline:
1. **Search space**: depth, channels, kernel size (3/5/7), block type (standard / depthwise-separable / inverted-residual), activation, input resolution
2. **Generator** builds a PyTorch network from each sampled configuration
3. **Bayesian optimisation (Optuna TPE)** proposes architectures
4. **Hardware profiler**: every candidate is exported to ONNX and its CPU latency, size and peak memory are **measured**; candidates that break the deployment constraints are **rejected before training**
5. Feasible candidates are trained briefly on CIFAR-10 and scored on a held-out validation split
6. The best architecture is saved and **trained from scratch**, then exported to ONNX and run with ONNX Runtime

> **Before running:** `Runtime -> Change runtime type -> T4 GPU`. The search resumes where it stopped if Colab disconnects.

## 1. Setup

In [ ]:
import os
REPO_URL = "https://github.com/sudoHasaan/edgeforge.git"
REPO_DIR = "/content/edgeforge"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!lscpu | grep -E "Model name|^CPU.s.:"

## 2. Persist outputs to Google Drive (recommended)

`checkpoints/`, `results/`, `onnx/` and `data/` are linked to `MyDrive/edgeforge_runs/`
(the same folder as Part 1, so all outputs end up together).

In [ ]:
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/edgeforge_runs"

if USE_DRIVE:
    import pathlib, shutil
    from google.colab import drive
    drive.mount("/content/drive")
    for d in ["checkpoints", "results", "onnx", "data"]:
        target = pathlib.Path(DRIVE_DIR) / d
        target.mkdir(parents=True, exist_ok=True)
        local = pathlib.Path(REPO_DIR) / d
        if local.is_symlink():
            continue
        if local.exists():
            shutil.copytree(local, target, dirs_exist_ok=True)
            shutil.rmtree(local)
        local.symlink_to(target, target_is_directory=True)
    print("Outputs are saved to", DRIVE_DIR)

## 3. Configuration

In [ ]:
N_TRIALS         = 40        # architectures proposed by Optuna (rejected ones are cheap)
CANDIDATE_EPOCHS = 5         # short training per feasible candidate
FINAL_EPOCHS     = 60        # training of the selected architecture from scratch
THREADS          = 1         # CPU threads for latency measurement (1 = single-core edge device)
OBJECTIVE        = "reward"  # "reward" = accuracy * (latency/limit)^-0.07 (MnasNet-style), or "accuracy"

# Deployment constraints. With USE_CALIBRATED=True they are set in step 4 from the median
# latency / size of random architectures measured on THIS machine's CPU.
USE_CALIBRATED = True
MAX_LATENCY_MS = 3.0         # ONNX Runtime CPU latency, batch 1
MAX_SIZE_MB    = 1.0         # ONNX model file size
MAX_MEM_MB     = 0           # peak inference memory limit (0 = record only)

## 4. Calibrate constraints on this CPU

Profiles 30 random architectures (no training) to see the latency and size range on this CPU.
The constraints are saved to `results/nas/constraints.json` and reused when the notebook is re-run, so the
search keeps the same constraints even if Colab gives you a different machine.

In [ ]:
import json, pathlib, pandas as pd
cfile = pathlib.Path("results/nas/constraints.json")

if cfile.exists():
    c = json.loads(cfile.read_text())
    print("Using saved constraints:", c)
else:
    if USE_CALIBRATED:
        !python scripts/nas_search.py --calibrate 30 --threads {THREADS}
        cal = pd.read_csv("results/nas/calibration.csv")
        MAX_LATENCY_MS = round(float(cal.latency_ms.median()), 2)
        MAX_SIZE_MB = round(float(cal.size_mb.median()), 2)
    c = {"max_latency_ms": MAX_LATENCY_MS, "max_size_mb": MAX_SIZE_MB, "max_mem_mb": MAX_MEM_MB}
    cfile.parent.mkdir(parents=True, exist_ok=True)
    cfile.write_text(json.dumps(c, indent=2))
    print("Constraints:", c)
MAX_LATENCY_MS, MAX_SIZE_MB, MAX_MEM_MB = c["max_latency_ms"], c["max_size_mb"], c["max_mem_mb"]

## 5. Run the search (then train the best architecture from scratch)

Each trial prints the architecture, its measured hardware cost, and either `REJECTED` (constraint violated)
or its validation accuracy. When the search ends, the best architecture is trained from scratch automatically.

In [ ]:
!python scripts/nas_search.py --trials {N_TRIALS} --epochs {CANDIDATE_EPOCHS} --threads {THREADS} \
    --max-latency-ms {MAX_LATENCY_MS} --max-size-mb {MAX_SIZE_MB} --max-mem-mb {MAX_MEM_MB} \
    --objective {OBJECTIVE} --final-epochs {FINAL_EPOCHS}

## 6. Search results

In [ ]:
hist = pd.read_csv("results/nas/search_history.csv")
print(hist.status.value_counts().to_string(), "\n")
cols = ["trial", "status", "val_top1", "objective", "latency_ms", "size_mb", "params", "macs_M",
        "inference_mem_mb", "arch", "reject_reason"]
hist[cols].sort_values("objective", ascending=False, na_position="last").head(15)

In [ ]:
print(json.dumps(json.loads(pathlib.Path("results/nas/best_architecture.json").read_text()), indent=2))

## 7. Export the NAS model to ONNX

In [ ]:
!python scripts/export_onnx.py --model nas

## 8. Inference: PyTorch and ONNX Runtime (CPU)

In [ ]:
!python scripts/infer.py --model nas --eval --samples 5

In [ ]:
!python scripts/onnx_infer.py --model nas --eval --latency --samples 5

## 9. Benchmark

Final NAS model (accuracy, size, latency, memory, constraint check) plus a deployment comparison of every
exported ONNX model, including the Part 1 students if they are in the same Drive folder.

In [ ]:
!python scripts/benchmark_nas.py --threads {THREADS}
display(pd.read_csv("results/nas/nas_benchmark.csv").T)
if pathlib.Path("results/deployment_benchmark.csv").exists():
    display(pd.read_csv("results/deployment_benchmark.csv"))

## 10. Report figures

In [ ]:
!python scripts/make_plots_nas.py
from IPython.display import Image, display
for f in ["nas_search_progress", "nas_accuracy_vs_latency", "nas_hardware_space",
          "nas_choice_effects", "nas_final_training"]:
    p = pathlib.Path(f"results/plots/{f}.png")
    if p.exists():
        display(Image(str(p)))

## 11. Download all outputs

Creates `edgeforge_q2_outputs.zip` (folder layout matches the repo).

In [ ]:
!cd {REPO_DIR} && rm -f /content/edgeforge_q2_outputs.zip && \
    zip -qr /content/edgeforge_q2_outputs.zip checkpoints/nas onnx/nas results -x "*_last.pth" "*.gitkeep"
!ls -lh /content/edgeforge_q2_outputs.zip
if USE_DRIVE:
    !cp /content/edgeforge_q2_outputs.zip {DRIVE_DIR}/
from google.colab import files
files.download("/content/edgeforge_q2_outputs.zip")